# 01 – Kaggle EDA and Baseline Model

This notebook builds on the ingestion pipeline in `pipelines/data_ingest.py` and the
processed dataset in `data/processed/transactions_clean.parquet`.

Goals for this notebook:

1. Frame the fraud detection problem and explain why **PR-AUC** is the primary metric.
2. Load the processed dataset (or fail gracefully with a clear message if missing).
3. Explore target imbalance and basic data characteristics.
4. Discuss potential leakage from each column.
5. Train a simple baseline model and report PR-AUC, ROC-AUC, and threshold-based metrics.
6. Inspect feature importance (permutation importance or model coefficients).
7. Explain decisions around sampling, metrics, and initial feature choices.

This notebook is part of **Step 1** and is intended to be runnable on a typical
developer laptop once the Kaggle dataset has been downloaded and processed.

## 1. Problem framing and metric choice (PR-AUC)

Online payments fraud detection has several important properties:

- **Extreme class imbalance**: very few transactions are fraudulent.
- **Asymmetric costs**: missing a fraud is more expensive than a false alarm, but too
  many false alarms damage user trust and operational capacity.
- **Evolving patterns**: adversaries adapt over time, so we care about how the model
  generalizes and how we monitor performance.

Because of this imbalance, we prefer **Precision–Recall AUC (PR-AUC)** over ROC-AUC as
the **primary** metric (see `context/05_METRICS_AND_EVAL.md`):

- ROC-AUC can look deceptively high on imbalanced data because true negatives dominate.
- PR-AUC gives a more meaningful view of performance on the positive (fraud) class.
- Threshold-based metrics like recall@precision and precision@recall are also critical
  for picking operating points that align with business requirements.

In this notebook, we will:

- Compute PR-AUC and ROC-AUC.
- Report recall at fixed precision and precision at fixed recall.
- Use these results to seed the evaluation strategy in later steps.

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance

PROCESSED_PATH = Path("../data/processed/transactions_clean.parquet")

if not PROCESSED_PATH.exists():
    raise FileNotFoundError(
        f"Processed parquet not found at {PROCESSED_PATH}.\n"
        "Run pipelines/data_ingest.py first (see README and scripts/kaggle_download.md)."
    )

df = pd.read_parquet(PROCESSED_PATH)
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,event_timestamp,customer_id,merchant_id,account_id,geo_cell_id,device_id
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0,2017-01-01 01:00:00+00:00,C1231006815,M1979787155,C1231006815,bef025d0b6f8738f,2aec01350b81e27e
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0,2017-01-01 01:00:00+00:00,C1666544295,M2044282225,C1666544295,4a60fac343a97baf,0d4bbd9d09068da8
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0,2017-01-01 01:00:00+00:00,C1305486145,C553264065,C1305486145,e92b5c96a213aae0,a622893342bf3c58
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0,2017-01-01 01:00:00+00:00,C840083671,C38997010,C840083671,75f26f60c8deafbd,df0ae76e3b253c51
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0,2017-01-01 01:00:00+00:00,C2048537720,M1230701703,C2048537720,2379793d5f5630f4,759f98e50fbb18f3


## 2. Target imbalance analysis

We start by looking at the prevalence of fraud (`isFraud`) and flagged transactions
(`isFlaggedFraud`). This gives us an idea of how imbalanced the dataset is and how
informative the existing flags might be.


In [2]:
fraud_rate = df["isFraud"].mean()
flagged_rate = df["isFlaggedFraud"].mean()

print(f"Fraud rate (isFraud=1): {fraud_rate:.6f}")
print(f"Flagged rate (isFlaggedFraud=1): {flagged_rate:.6f}")

df[["type", "isFraud", "isFlaggedFraud"]].groupby("type").mean().sort_values(
    by="isFraud", ascending=False
)

Fraud rate (isFraud=1): 0.000735
Flagged rate (isFlaggedFraud=1): 0.000000


,isFraud,isFlaggedFraud
type,,
TRANSFER,0.004277,0.0
CASH_OUT,0.001128,0.0
CASH_IN,0.000000,0.0
DEBIT,0.000000,0.0
PAYMENT,0.000000,0.0


## 3. Leakage discussion

Potential leakage sources to consider:

- **Balances (`oldbalanceOrg`, `newbalanceOrig`, `oldbalanceDest`, `newbalanceDest`)**:
  - We must interpret them carefully. For example, if `newbalanceOrig` reflects a
    post-transaction state that depends on whether the transaction was reversed,
    then it may encode label information.
- **`isFlaggedFraud`**:
  - This is a pre-existing flag that might be based on rules or previous models. We
    should treat it as a feature but keep in mind that it encodes prior knowledge.
- **`step`**:
  - We will use it to construct `event_timestamp`, but train/validation/test splits
    should respect time ordering to avoid leakage from the future.

In Step 1, we keep the feature set intentionally simple and will revisit leakage in
more detail in later iterations.

## 4. Baseline feature set and train/validation split

We define a small baseline feature set using only numeric columns that are straightforward
to compute and reason about. This keeps the first model simple and debuggable.

For this baseline, we will use:

- `amount`
- `oldbalanceOrg`
- `newbalanceOrig`
- `oldbalanceDest`
- `newbalanceDest`
- `isFlaggedFraud`

We also encode transaction `type` as one-hot indicators.

For the train/test split, we use a simple holdout with stratification, but in later
steps we will move to strictly time-based splits (see `context/05_METRICS_AND_EVAL.md`).

In [3]:
feature_cols_numeric = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "isFlaggedFraud",
]

df_model = df.copy()
df_model = pd.get_dummies(df_model, columns=["type"], drop_first=True)

feature_cols = feature_cols_numeric + [c for c in df_model.columns if c.startswith("type_")]
X = df_model[feature_cols].fillna(0.0)
y = df_model["isFraud"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

X_train.shape, X_test.shape

((160000, 10), (40000, 10))

## 5. Baseline model and metrics

We train a simple logistic regression model and evaluate PR-AUC, ROC-AUC, and threshold-based
metrics. The goal here is not to optimize performance but to establish a reproducible baseline
and show how metrics will be reported.


In [4]:
model = LogisticRegression(max_iter=1000, n_jobs=-1)
model.fit(X_train, y_train)

y_score = model.predict_proba(X_test)[:, 1]

pr_auc = average_precision_score(y_test, y_score)
roc_auc = roc_auc_score(y_test, y_score)
print(f"PR-AUC:  {pr_auc:.4f}")
print(f"ROC-AUC: {roc_auc:.4f}")

precisions, recalls, thresholds = precision_recall_curve(y_test, y_score)

def recall_at_precision(target_precision: float) -> float:
    mask = precisions >= target_precision
    if not np.any(mask):
        return 0.0
    return float(recalls[mask].max())

def precision_at_recall(target_recall: float) -> float:
    mask = recalls >= target_recall
    if not np.any(mask):
        return 0.0
    return float(precisions[mask].max())

print(f"Recall @ precision>=0.95: {recall_at_precision(0.95):.4f}")
print(f"Precision @ recall>=0.80: {precision_at_recall(0.80):.4f}")

PR-AUC:  0.2385
ROC-AUC: 0.9656
Recall @ precision>=0.95: 0.1034
Precision @ recall>=0.80: 0.1064


## 6. Feature importance

To get a first sense of which features matter, we compute permutation importance on the
held-out test set. This is relatively slow but gives a more robust sense of importance
than raw model coefficients alone.


In [5]:
result = permutation_importance(
    model,
    X_test,
    y_test,
    n_repeats=5,
    random_state=42,
    n_jobs=-1,
)

importances = pd.Series(result.importances_mean, index=feature_cols).sort_values(
    ascending=False
)
importances.head(20)

newbalanceOrig    0.145025
oldbalanceOrg     0.143810
amount            0.003160
oldbalanceDest    0.000020
newbalanceDest    0.000005
isFlaggedFraud    0.000000
type_CASH_OUT     0.000000
type_DEBIT        0.000000
type_PAYMENT      0.000000
type_TRANSFER     0.000000
dtype: float64

## 7. Decisions explained

This section summarizes the key decisions made in Step 1 and how they tie back to
the broader project context (`context/*.md`):

- **Sampling**:
  - We use a configurable `--sample_rows` in `pipelines/data_ingest.py` (defaulting to
    200k rows) to keep experiments lightweight and free-tier friendly.
  - The pipeline is chunk-based so that we do not need to load the full CSV into memory.

- **Entity IDs**:
  - `customer_id` = `nameOrig`
  - `merchant_id` = `nameDest`
  - `account_id` = `nameOrig`
  - `geo_cell_id` = deterministic hash of `nameDest`
  - `device_id` = deterministic hash of `nameOrig + '|' + nameDest`
  - These mappings are documented in `context/04_DATASET_PLAN.md` and used by
    `pipelines/data_ingest.py` and `pipelines/build_entity_tables.py`.

- **Metrics**:
  - We emphasize PR-AUC as the main metric (see `context/05_METRICS_AND_EVAL.md`).
  - We also compute ROC-AUC, recall@precision, and precision@recall to inform
    threshold selection.

- **Feature choices** (baseline):
  - Numeric balances and amount are straightforward to compute online and serve as
    a simple starting point.
  - `isFlaggedFraud` is included as a proxy for existing rule-based systems.
  - Encoded transaction `type` features capture different behavioral patterns.

These decisions will be revisited and expanded in later steps as we move towards
a richer Feast feature repository and real-time online serving via Kafka + Postgres.